## 라이브러리 임포트

In [1]:
import numpy as np
from tqdm import tqdm
import os
import matplotlib.pyplot as plt

## 재현성을 위한 시드 설정

In [2]:
# 재현성을 위한 시드 설정
SEED = 42
np.random.seed(SEED)

print(f"✓ 시드 설정 완료: SEED={SEED}")

✓ 시드 설정 완료: SEED=42


## 색상 팔레트 정의

In [3]:
# 무지개 색상 팔레트 정의 (7가지 색상)
RAINBOW_COLORS = [
    (255, 0, 0),    # 빨강
    (255, 127, 0),  # 주황
    (255, 255, 0),  # 노랑  
    (0, 255, 0),    # 초록
    (0, 0, 255),    # 파랑
    (75, 0, 130),   # 남색 (인디고)
    (148, 0, 211)   # 보라
]

RAINBOW_NAMES = [
    'RED',
    'ORANGE',
    'YELLOW',
    'GREEN',
    'BLUE',
    'INDIGO',
    'VIOLET'
]

print(f"✓ 색상 팔레트: {len(RAINBOW_COLORS)}가지 무지개 색상")
for name, color in zip(RAINBOW_NAMES, RAINBOW_COLORS):
    print(f"  {name:8s}: RGB{color}")

✓ 색상 팔레트: 7가지 무지개 색상
  RED     : RGB(255, 0, 0)
  ORANGE  : RGB(255, 127, 0)
  YELLOW  : RGB(255, 255, 0)
  GREEN   : RGB(0, 255, 0)
  BLUE    : RGB(0, 0, 255)
  INDIGO  : RGB(75, 0, 130)
  VIOLET  : RGB(148, 0, 211)


## 색상 선택 함수

In [4]:
def get_random_rainbow_color():
    """무지개 팔레트에서 임의의 색상을 선택합니다."""
    idx = np.random.randint(0, len(RAINBOW_COLORS))
    return RAINBOW_COLORS[idx], RAINBOW_NAMES[idx]

## 색상 적용 함수

In [5]:
def colorize_digit_with_threshold(gray_array, fg_color, bg_color, threshold=64):
    """
    그레이스케일 배열에 임계값을 적용하여 경계선 번짐 없이 선명한 전경/배경 색을 적용합니다.
    
    Args:
        gray_array (numpy.ndarray): 그레이스케일 이미지 배열 (28x28)
        fg_color (tuple): 전경 색상 RGB
        bg_color (tuple): 배경 색상 RGB
        threshold (int): 임계값 (기본값: 128)
    
    Returns:
        numpy.ndarray: RGB 색상 이미지 배열 (28x28x3)
    """
    # RGB 이미지 생성 (배경색으로 초기화)
    rgb_image = np.zeros((*gray_array.shape, 3), dtype=np.uint8)
    rgb_image[:, :] = bg_color
    
    # 임계값을 적용하여 마스크 생성
    mask = gray_array > threshold
    
    # 마스크에 해당하는 픽셀을 전경색으로 설정
    rgb_image[mask] = fg_color
    
    return rgb_image

## 데이터셋 색상화 함수

In [6]:
def colorize_mnist_dataset(input_path, output_path):
    """
    MNIST 흑백 데이터셋을 로드하여 각 이미지에 랜덤 색상을 적용한 RGB 데이터셋 생성
    
    Args:
        input_path (str): 입력 .npz 파일 경로 (mnist_train.npz)
        output_path (str): 출력 .npz 파일 경로 (mnist_colormod.npz)
    """
    print("="*60)
    print("MNIST 색상 적용 도구")
    print("="*60)
    
    # 1. 원본 데이터 로드
    print(f"\n📂 데이터 로드 중: {input_path}")
    data = np.load(input_path)
    
    if 'train_images' not in data or 'train_labels' not in data:
        raise ValueError(f"❌ 필수 키가 없습니다. 필요: 'train_images', 'train_labels'\n실제: {list(data.keys())}")
    
    images = data['train_images']
    labels = data['train_labels']
    
    print(f"✅ 로드 완료")
    print(f"   이미지: {images.shape}, dtype: {images.dtype}")
    print(f"   레이블: {labels.shape}, dtype: {labels.dtype}")
    
    # 2. 색상 적용
    print(f"\n🎨 색상 적용 중...")
    print(f"   총 {len(images)}개 이미지 처리")
    print(f"   색상 팔레트: {len(RAINBOW_COLORS)}가지 무지개 색상")
    
    colored_images = []
    
    for i in tqdm(range(len(images)), desc="색상 적용"):
        # 전경과 배경 색상을 무작위로 선택
        fg_color, _ = get_random_rainbow_color()
        bg_color, _ = get_random_rainbow_color()
        
        # 전경과 배경 색상이 같지 않도록 보장
        while bg_color == fg_color:
            bg_color, _ = get_random_rainbow_color()
        
        # 색상 적용
        colored_img = colorize_digit_with_threshold(images[i], fg_color, bg_color)
        colored_images.append(colored_img)
    
    # numpy 배열로 변환
    colored_images = np.array(colored_images, dtype=np.uint8)
    
    print(f"\n✅ 색상 적용 완료!")
    print(f"   변환된 이미지: {colored_images.shape}")
    print(f"   레이블: {labels.shape}")
    
    # 3. 저장
    print(f"\n💾 저장 중: {output_path}")
    np.savez_compressed(
        output_path,
        train_images=colored_images,
        train_labels=labels
    )
    
    file_size_mb = os.path.getsize(output_path) / 1024 / 1024
    
    print(f"\n{'='*60}")
    print(f"✓ 데이터셋 저장 완료!")
    print(f"{'='*60}")
    print(f"파일 경로: {output_path}")
    print(f"파일 크기: {file_size_mb:.2f} MB")
    print(f"이미지: {colored_images.shape} (RGB 3채널)")
    print(f"레이블: {labels.shape}")
    print(f"{'='*60}")
    
    return colored_images, labels

## 결과 시각화 함수

In [7]:
def visualize_samples(images, labels, num_samples=10):
    """
    색상이 적용된 샘플 이미지들을 시각화합니다.
    
    Args:
        images (numpy.ndarray): RGB 이미지 배열
        labels (numpy.ndarray): 레이블 배열
        num_samples (int): 표시할 샘플 개수
    """
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle('색상이 적용된 MNIST 샘플', fontsize=16, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        if i < num_samples:
            ax.imshow(images[i])
            ax.set_title(f'Label: {labels[i]}', fontsize=12)
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## 파일 경로 설정

In [ ]:
# 파일 경로 설정
input_path = '../assets/mnist_train.npz'
output_path = '../assets/mnist_colormod.npz'

# 현재 노트북 디렉토리 기준으로 경로 설정
notebook_dir = os.getcwd()
input_path = os.path.join(notebook_dir, input_path)
output_path = os.path.join(notebook_dir, output_path)

print(f"입력 파일: {input_path}")
print(f"출력 파일: {output_path}")

입력 파일: c:\Users\lota\Documents\SourceCodes\ML_25_2\data_augmentation\mnist_train.npz
출력 파일: c:\Users\lota\Documents\SourceCodes\ML_25_2\data_augmentation\mnist_colormod.npz


## 실행: 색상 적용 및 저장

In [9]:
# 색상 적용 및 저장 실행
try:
    colored_images, labels = colorize_mnist_dataset(input_path, output_path)
    print("\n✅ 모든 작업 완료!")
    
except FileNotFoundError:
    print(f"\n❌ 오류: {input_path} 파일을 찾을 수 없습니다.")
    print(f"   현재 디렉토리: {notebook_dir}")
    
except Exception as e:
    print(f"\n❌ 오류 발생: {e}")
    raise

MNIST 색상 적용 도구

📂 데이터 로드 중: c:\Users\lota\Documents\SourceCodes\ML_25_2\data_augmentation\mnist_train.npz

❌ 오류: c:\Users\lota\Documents\SourceCodes\ML_25_2\data_augmentation\mnist_train.npz 파일을 찾을 수 없습니다.
   현재 디렉토리: c:\Users\lota\Documents\SourceCodes\ML_25_2\data_augmentation


## 결과 시각화

In [10]:
# 생성된 색상 이미지 샘플 시각화
if 'colored_images' in locals() and 'labels' in locals():
    visualize_samples(colored_images, labels, num_samples=10)
else:
    print("먼저 색상 적용을 실행해주세요.")

먼저 색상 적용을 실행해주세요.
